# Prophet Baseline

Đánh giá Prophet trên toàn bộ 100 series (Store × Product), theo cùng convention với `run_naive.py` và `run_lstm2.py`.

In [1]:
import numpy as np
import pandas as pd
import warnings
import sys, os
from prophet import Prophet

warnings.filterwarnings('ignore')
sys.path.append('..')

In [2]:
import tensorflow as tf
import sys, os
import glob
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

tf.random.set_seed(42)
np.random.seed(42)

if os.path.exists("/kaggle/input"):

    matches = glob.glob(
        "/kaggle/input/**/sales_data.csv",
        recursive=True
    )

    DATA_PATH = matches[0] if matches else "/kaggle/input/sales_data.csv"
    RESULT_DIR = "/kaggle/working"

else:

    DATA_PATH  = "/Users/P837032/Daily/Model/dataset/sales_data.csv"
    RESULT_DIR = "/Users/P837032/Daily/Model/result"


print("DATA_PATH:", DATA_PATH)

TARGET     = 'Units Sold'
TRAIN_END  = '2023-06-30'
VAL_END    = '2023-10-31'
HORIZONS   = [7, 14, 28]
LOOKBACK   = 30
LAG        = 7

2026-03-30 13:00:27.010064: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774875627.299108      54 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774875627.386343      54 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774875628.119396      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774875628.119449      54 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774875628.119453      54 computation_placer.cc:177] computation placer alr

DATA_PATH: /kaggle/input/datasets/thaonngyn/retail-data/sales_data.csv


In [3]:
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store ID', 'Product ID', 'Date'])

series_dict = {
    f"{store}_{product}": grp.set_index('Date')[TARGET]
    for (store, product), grp in df.groupby(['Store ID', 'Product ID'])
}
series_ids = sorted(series_dict.keys())
print(f'Total series: {len(series_ids)}')

Total series: 100


In [4]:
def prophet_fn(train_dates, train_values, horizon):
    df_fit = pd.DataFrame({'ds': train_dates, 'y': train_values.astype(float)})
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
    m.fit(df_fit)
    future = m.make_future_dataframe(periods=horizon, freq='D', include_history=False)
    fc = m.predict(future)['yhat'].values
    return np.clip(fc, 0, None)


def rolling_eval(series, split_end, horizon):
    eval_start = pd.Timestamp(split_end) + pd.Timedelta(days=1)
    eval_end   = series.index.max()
    all_fc, all_ac, all_trains = [], [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        train_s = series[:t - pd.Timedelta(days=1)]
        actual  = series[t: t + pd.Timedelta(days=horizon - 1)].values.astype(float)
        if len(train_s) < 2 or len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue
        all_fc.append(prophet_fn(train_s.index, train_s.values, horizon))
        all_ac.append(actual)
        all_trains.append(train_s.values.astype(float))
        t += pd.Timedelta(days=horizon)
    if not all_fc:
        return {k: np.nan for k in ['mse', 'rmse', 'mae', 'mape', 'smape', 'mase', 'rmsle']}
    fc_arr = np.array(all_fc)
    ac_arr = np.array(all_ac)
    fc_c   = np.clip(fc_arr, 0, None)
    ac_c   = np.clip(ac_arr, 0, None)
    # MASE denominator: naive lag-7 on last training window
    lag = min(7, len(all_trains[-1]) - 1)
    tr  = all_trains[-1]
    denom = np.mean(np.abs(tr[lag:] - tr[:-lag])) or 1.0
    return {
        'mse'  : float(np.mean((fc_arr - ac_arr) ** 2)),
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'mae'  : float(np.mean(np.abs(fc_arr - ac_arr))),
        'mape' : float(np.mean(np.abs((fc_arr - ac_arr) / (ac_arr + 1e-8))) * 100),
        'smape': float((2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100),
        'mase' : float(np.mean(np.abs(fc_arr - ac_arr)) / denom),
        'rmsle': float(np.sqrt(np.mean((np.log1p(fc_c) - np.log1p(ac_c)) ** 2))),
    }

In [5]:
os.makedirs(RESULT_DIR, exist_ok=True)
import warnings
warnings.filterwarnings("ignore")
results = []
details = []

for h in HORIZONS:
    scores = {k: [] for k in ['mse', 'rmse', 'mae', 'mape', 'smape', 'mase', 'rmsle']}
    for i, sid in enumerate(series_ids):
        store, product = sid.split('_', 1)
        r = rolling_eval(series_dict[sid], VAL_END, h)
        for k in scores:
            scores[k].append(r[k])
        details.append({'model': 'Prophet', 'store': store, 'product': product, 'horizon': h, **{k: round(float(v), 4) for k, v in r.items()}})
        if (i + 1) % 20 == 0:
            print(f'  H={h} | {i+1}/{len(series_ids)} done')
    row = {
        'model': 'Prophet', 'dataset': 'retail_inventory_daily', 'target': TARGET,
        'horizon': h,
        'mean_mse':    round(float(np.nanmean(scores['mse'])),    4),
        'mean_rmse':   round(float(np.nanmean(scores['rmse'])),   4),
        'mean_mae':    round(float(np.nanmean(scores['mae'])),    4),
        'mean_mape':   round(float(np.nanmean(scores['mape'])),   4),
        'mean_smape':  round(float(np.nanmean(scores['smape'])),  4),
        'median_smape':round(float(np.nanmedian(scores['smape'])),4),
        'mean_mase':   round(float(np.nanmean(scores['mase'])),   4),
        'median_mase': round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmsle':  round(float(np.nanmean(scores['rmsle'])),  4),
        'median_rmsle':round(float(np.nanmedian(scores['rmsle'])),4),
    }
    results.append(row)
    print(f"H={h:2d} | RMSE={row['mean_rmse']:.2f} sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSLE={row['mean_rmsle']:.4f}")

pd.DataFrame(results).to_csv(f'{RESULT_DIR}/prophet_daily_summary.csv', index=False)
pd.DataFrame(details).to_csv(f'{RESULT_DIR}/prophet_daily_details.csv', index=False)
print(f'\nSaved to {RESULT_DIR}/prophet_daily_summary.csv')

13:00:57 - cmdstanpy - INFO - Chain [1] start processing
13:00:57 - cmdstanpy - INFO - Chain [1] done processing
13:00:58 - cmdstanpy - INFO - Chain [1] start processing
13:00:58 - cmdstanpy - INFO - Chain [1] done processing
13:00:58 - cmdstanpy - INFO - Chain [1] start processing
13:00:58 - cmdstanpy - INFO - Chain [1] done processing
13:00:58 - cmdstanpy - INFO - Chain [1] start processing
13:00:58 - cmdstanpy - INFO - Chain [1] done processing
13:00:58 - cmdstanpy - INFO - Chain [1] start processing
13:00:58 - cmdstanpy - INFO - Chain [1] done processing
13:00:58 - cmdstanpy - INFO - Chain [1] start processing
13:00:58 - cmdstanpy - INFO - Chain [1] done processing
13:00:59 - cmdstanpy - INFO - Chain [1] start processing
13:00:59 - cmdstanpy - INFO - Chain [1] done processing
13:00:59 - cmdstanpy - INFO - Chain [1] start processing
13:00:59 - cmdstanpy - INFO - Chain [1] done processing
13:00:59 - cmdstanpy - INFO - Chain [1] start processing
13:00:59 - cmdstanpy - INFO - Chain [1]

  H=7 | 20/100 done


13:01:19 - cmdstanpy - INFO - Chain [1] start processing
13:01:19 - cmdstanpy - INFO - Chain [1] done processing
13:01:19 - cmdstanpy - INFO - Chain [1] start processing
13:01:19 - cmdstanpy - INFO - Chain [1] done processing
13:01:19 - cmdstanpy - INFO - Chain [1] start processing
13:01:19 - cmdstanpy - INFO - Chain [1] done processing
13:01:19 - cmdstanpy - INFO - Chain [1] start processing
13:01:19 - cmdstanpy - INFO - Chain [1] done processing
13:01:19 - cmdstanpy - INFO - Chain [1] start processing
13:01:19 - cmdstanpy - INFO - Chain [1] done processing
13:01:20 - cmdstanpy - INFO - Chain [1] start processing
13:01:20 - cmdstanpy - INFO - Chain [1] done processing
13:01:20 - cmdstanpy - INFO - Chain [1] start processing
13:01:20 - cmdstanpy - INFO - Chain [1] done processing
13:01:20 - cmdstanpy - INFO - Chain [1] start processing
13:01:20 - cmdstanpy - INFO - Chain [1] done processing
13:01:20 - cmdstanpy - INFO - Chain [1] start processing
13:01:20 - cmdstanpy - INFO - Chain [1]

  H=7 | 40/100 done


13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1] done processing
13:01:40 - cmdstanpy - INFO - Chain [1] start processing
13:01:40 - cmdstanpy - INFO - Chain [1]

...

  H=28 | 60/100 done


13:03:42 - cmdstanpy - INFO - Chain [1] start processing
13:03:42 - cmdstanpy - INFO - Chain [1] done processing
13:03:42 - cmdstanpy - INFO - Chain [1] start processing
13:03:42 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1] done processing
13:03:43 - cmdstanpy - INFO - Chain [1] start processing
13:03:43 - cmdstanpy - INFO - Chain [1]

  H=28 | 80/100 done


13:03:47 - cmdstanpy - INFO - Chain [1] done processing
13:03:47 - cmdstanpy - INFO - Chain [1] start processing
13:03:47 - cmdstanpy - INFO - Chain [1] done processing
13:03:47 - cmdstanpy - INFO - Chain [1] start processing
13:03:47 - cmdstanpy - INFO - Chain [1] done processing
13:03:47 - cmdstanpy - INFO - Chain [1] start processing
13:03:47 - cmdstanpy - INFO - Chain [1] done processing
13:03:47 - cmdstanpy - INFO - Chain [1] start processing
13:03:47 - cmdstanpy - INFO - Chain [1] done processing
13:03:48 - cmdstanpy - INFO - Chain [1] start processing
13:03:48 - cmdstanpy - INFO - Chain [1] done processing
13:03:48 - cmdstanpy - INFO - Chain [1] start processing
13:03:48 - cmdstanpy - INFO - Chain [1] done processing
13:03:48 - cmdstanpy - INFO - Chain [1] start processing
13:03:48 - cmdstanpy - INFO - Chain [1] done processing
13:03:48 - cmdstanpy - INFO - Chain [1] start processing
13:03:48 - cmdstanpy - INFO - Chain [1] done processing
13:03:48 - cmdstanpy - INFO - Chain [1] 

  H=28 | 100/100 done
H=28 | RMSE=39.97 sMAPE=38.47% MASE=0.7627 RMSLE=0.6095

Saved to /kaggle/working/prophet_daily_summary.csv


In [6]:
pd.DataFrame(results)

,model,dataset,target,horizon,mean_mse,mean_rmse,mean_mae,mean_mape,mean_smape,median_smape,mean_mase,median_mase,mean_rmsle,median_rmsle
0,Prophet,retail_inventory_daily,Units Sold,7,1610.2865,39.5931,31.3358,4.554659e+09,37.7751,38.2383,0.7518,0.7526,0.6004,0.5786
1,Prophet,retail_inventory_daily,Units Sold,14,1636.2751,39.9215,31.6921,4.534711e+09,38.4222,38.7468,0.7616,0.7547,0.6083,0.5842
2,Prophet,retail_inventory_daily,Units Sold,28,1641.0446,39.9726,31.7497,4.576699e+09,38.4665,38.8115,0.7627,0.7537,0.6095,0.5857
